# Blood Vessel Segmentation with Segment Anything (SAM)

This notebook fine-tunes Meta's Segment Anything Model (SAM) for blood-vessel-like microscopy images.  
The workflow is split into two parts:

1. prepare image and mask patches for training
2. load the trained model and apply it to a larger test image

The notebook was written for Google Colab. The paths point to Google Drive folders and should be changed if the data is stored somewhere else.


## 1. Environment setup

The first cell installs the packages used in this notebook. In Colab, it is usually best to run this once after starting a fresh runtime.  
For training, select a GPU runtime first: **Runtime → Change runtime type → GPU**.


In [ ]:
# Install the packages needed for the SAM training workflow.
# The NumPy version is pinned because some image-processing packages can break
# when Colab pulls a newer NumPy release automatically.

!pip -q install -U pip setuptools wheel
!pip -q install "numpy>=2.1,<2.2"
!pip -q install "scipy>=1.13" matplotlib
!pip -q install "opencv-python-headless>=4.9.0.80"

# Segment Anything, Hugging Face tools, data handling, and the segmentation loss.
!pip -q install git+https://github.com/facebookresearch/segment-anything.git
!pip -q install git+https://github.com/huggingface/transformers.git
!pip -q install datasets monai tifffile

# Installed without dependencies so it does not force an older NumPy version.
!pip -q install --no-deps patchify


In [ ]:
# Import the main packages and print versions.
# This makes it easier to debug the notebook later if Colab changes its default environment.

import numpy as np
import scipy
import cv2
from scipy import ndimage
from patchify import patchify
import tifffile
import matplotlib.pyplot as plt

print("numpy:", np.__version__)
print("scipy:", scipy.__version__)
print("opencv:", cv2.__version__)
print("all imports ok")


In [ ]:
# Check whether PyTorch sees the GPU.
# Training still works on CPU, but it will be much slower.

import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


## 2. Load microscopy images and masks

The training images and masks are stored as TIFF stacks.  
Expected format in this notebook:

- images: `(N, H, W, 3)` RGB microscopy images
- masks: `(N, H, W)` labelled or binary masks

If the masks contain more than one foreground label, they are converted to a simple binary mask later.


In [ ]:
# Mount Google Drive so Colab can access the image and mask files.

from google.colab import drive

drive.mount("/content/drive")


In [ ]:
# Paths to the training data.
# Change these paths if the files are stored in another Drive folder.

images_path = "/content/drive/MyDrive/Colab/microscopy.tif"
masks_path  = "/content/drive/MyDrive/Colab/masks.tif"


In [ ]:
# Load the TIFF stacks and print basic information.
# This is a quick check that the files are found and that image/mask dimensions match.

import os
import tifffile

print("images exists:", os.path.exists(images_path))
print("masks exists:", os.path.exists(masks_path))

imgs = tifffile.imread(images_path)
msks = tifffile.imread(masks_path)

print("imgs shape:", imgs.shape, imgs.dtype)
print("msks shape:", msks.shape, msks.dtype)


In [ ]:
# Inspect mask values.
# If the mask contains labels such as 0, 1, and 2, everything above 0 is treated as vessel/foreground.

import numpy as np

print("mask unique values (sample):", np.unique(msks[0])[:20])
print("mask max:", msks.max())


In [ ]:
# Convert the masks to binary format: 0 = background, 1 = vessel/foreground.

msks_bin = (msks > 0).astype(np.uint8)
print("binary unique:", np.unique(msks_bin))


## 3. Create training patches

SAM is trained here on `256 × 256` patches.  
The image patches and mask patches are cut in the same order, so each image patch keeps its matching ground-truth mask.


In [ ]:
# Split the full images and masks into 256 x 256 patches.
# step = 256 means there is no overlap between neighbouring patches.

import numpy as np
from patchify import patchify

patch_size = 256
step = 256

# Image patches: expected input shape is (N, H, W, 3).
all_img_patches = []
for idx in range(imgs.shape[0]):
    large_image = imgs[idx]

    # patchify returns an extra singleton dimension for RGB images.
    patches_img = patchify(large_image, (patch_size, patch_size, 3), step=step)

    for i in range(patches_img.shape[0]):
        for j in range(patches_img.shape[1]):
            patch = patches_img[i, j, 0, :, :, :]
            all_img_patches.append(patch)

images = np.array(all_img_patches, dtype=np.uint8)
print("image patches:", images.shape, images.dtype)


# Mask patches: convert each patch to binary foreground/background.
all_mask_patches = []
for idx in range(msks.shape[0]):
    large_mask = msks[idx]
    patches_mask = patchify(large_mask, (patch_size, patch_size), step=step)

    for i in range(patches_mask.shape[0]):
        for j in range(patches_mask.shape[1]):
            patch = patches_mask[i, j, :, :]
            patch = (patch > 0).astype(np.uint8)
            all_mask_patches.append(patch)

masks = np.array(all_mask_patches, dtype=np.uint8)
print("mask patches:", masks.shape, masks.dtype)
print("mask unique:", np.unique(masks))


In [ ]:
# Remove almost-empty patches.
# Empty background patches slow down training and can make the model less focused on vessel regions.
# The threshold can be adjusted if too many or too few patches are kept.

foreground_fraction = masks.reshape(masks.shape[0], -1).mean(axis=1)
keep = foreground_fraction > 0.0005

images = images[keep]
masks  = masks[keep]

print("kept patches:", images.shape[0])


In [ ]:
# Visual checkpoint: show one random patch, its mask, and an overlay.
# This catches many common problems early, such as shifted masks or wrong foreground values.

import matplotlib.pyplot as plt
import random

k = random.randrange(images.shape[0])

plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.title("Image")
plt.imshow(images[k])
plt.axis("off")

plt.subplot(1, 3, 2)
plt.title("Mask")
plt.imshow(masks[k], cmap="gray")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.title("Overlay")
plt.imshow(images[k])
plt.imshow(masks[k], cmap="jet", alpha=0.35)
plt.axis("off")

plt.show()


In [ ]:
# Keep only patches that still contain foreground after filtering.
# This second check is redundant in most cases, but it keeps the later dataset clean.

valid_indices = [i for i, mask in enumerate(masks) if mask.max() != 0]
filtered_images = images[valid_indices]
filtered_masks  = masks[valid_indices]

print("Image shape:", filtered_images.shape)
print("Mask shape:", filtered_masks.shape)


## 4. Convert patches to a Hugging Face dataset

The SAM processor expects PIL images.  
Here the NumPy arrays are converted into PIL images and wrapped in a small Hugging Face `Dataset`.


In [ ]:
# Build a small Hugging Face dataset with image/mask pairs.
# Masks are stored as 0/255 grayscale images because this is easy to inspect visually.

from datasets import Dataset
from PIL import Image
import numpy as np

dataset_dict = {
    "image": [Image.fromarray(img, mode="RGB") for img in filtered_images],
    "label": [
        Image.fromarray((mask * 255).astype(np.uint8), mode="L")
        for mask in filtered_masks
    ],
}

dataset = Dataset.from_dict(dataset_dict)
print(dataset)


In [ ]:
# Another quick visual check, this time after conversion to the Hugging Face dataset.

import matplotlib.pyplot as plt
import random

i = random.randrange(len(dataset))

plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
plt.title("Image")
plt.imshow(dataset[i]["image"])
plt.axis("off")

plt.subplot(1, 2, 2)
plt.title("Mask")
plt.imshow(dataset[i]["label"], cmap="gray")
plt.axis("off")

plt.show()


## 5. Bounding-box prompts for SAM

SAM needs a prompt.  
For training, the prompt is generated automatically from the ground-truth mask by drawing a bounding box around the foreground region. A small random jitter is added so the model does not only learn perfectly tight boxes.


In [ ]:
# Create a bounding box around the foreground mask.
# The returned box uses SAM's format: [x_min, y_min, x_max, y_max].

import numpy as np

def get_bounding_box(ground_truth_map, max_jitter=20):
    """
    Return a slightly jittered bounding box around the foreground mask.

    Parameters
    ----------
    ground_truth_map : np.ndarray
        Binary mask with foreground values > 0.
    max_jitter : int
        Maximum number of pixels added/subtracted from each box side.

    Returns
    -------
    list[int] or None
        Bounding box in [x_min, y_min, x_max, y_max] format.
    """
    ys, xs = np.where(ground_truth_map > 0)
    if len(xs) == 0 or len(ys) == 0:
        return None

    x_min, x_max = xs.min(), xs.max()
    y_min, y_max = ys.min(), ys.max()
    H, W = ground_truth_map.shape

    # Add random margin, while keeping the box inside the image.
    x_min = max(0, x_min - np.random.randint(0, max_jitter + 1))
    x_max = min(W - 1, x_max + np.random.randint(0, max_jitter + 1))
    y_min = max(0, y_min - np.random.randint(0, max_jitter + 1))
    y_max = min(H - 1, y_max + np.random.randint(0, max_jitter + 1))

    return [int(x_min), int(y_min), int(x_max), int(y_max)]


In [ ]:
# Check one generated bounding box before using it in the dataset class.

bbox = get_bounding_box(filtered_masks[0])
print("bbox:", bbox)


## 6. Dataset class for SAM fine-tuning

This wrapper prepares one sample at a time for the SAM model.  
For each patch it returns the processed image, the box prompt, and the corresponding ground-truth mask used for the loss calculation.


In [ ]:
# PyTorch dataset for SAM fine-tuning.
# Each item contains:
# - the processed RGB image
# - a bounding-box prompt
# - the binary ground-truth mask

import numpy as np
import torch
from torch.utils.data import Dataset

class SAMDataset(Dataset):
    def __init__(self, dataset, processor):
        self.dataset = dataset
        self.processor = processor

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        image = item["image"]
        ground_truth_mask = np.array(item["label"])

        # Convert 0/255 mask back to 0/1 for training.
        ground_truth_mask = (ground_truth_mask > 0).astype(np.uint8)

        # Use the ground-truth mask to create the box prompt.
        prompt = get_bounding_box(ground_truth_mask)

        # Processor handles resizing, normalization, and prompt formatting for SAM.
        inputs = self.processor(
            image,
            input_boxes=[[prompt]],
            return_tensors="pt"
        )

        # Remove the artificial batch dimension added by the processor.
        inputs = {k: v.squeeze(0) for k, v in inputs.items()}

        # Keep boxes as float tensors. Some SAM/transformers versions are strict here.
        inputs["input_boxes"] = inputs["input_boxes"].float()
        inputs["ground_truth_mask"] = torch.from_numpy(
            ground_truth_mask
        ).unsqueeze(0).float()

        return inputs


In [ ]:
# Load SAM and inspect one processed training sample.
# The base model is used here; only the mask decoder will be trained later.

from transformers import SamProcessor, SamModel
import torch

checkpoint = "facebook/sam-vit-base"
# checkpoint = "facebook/sam-vit-large"
# checkpoint = "facebook/sam-vit-huge"

processor = SamProcessor.from_pretrained(checkpoint)
model = SamModel.from_pretrained(checkpoint)
model.to("cuda" if torch.cuda.is_available() else "cpu")

sam_dataset = SAMDataset(dataset, processor)
sample = sam_dataset[0]

for k, v in sample.items():
    if hasattr(v, "shape"):
        print(k, v.shape, v.dtype)
    else:
        print(k, type(v))


In [ ]:
# Build the DataLoader used for training and inspect one batch.

from transformers import SamProcessor
from torch.utils.data import DataLoader

processor = SamProcessor.from_pretrained("facebook/sam-vit-base")
train_dataset = SAMDataset(dataset=dataset, processor=processor)

example = train_dataset[0]
for k, v in example.items():
    if hasattr(v, "shape"):
        print(k, v.shape, v.dtype)
    else:
        print(k, type(v))

train_dataloader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    drop_last=False
)

batch = next(iter(train_dataloader))
for k, v in batch.items():
    print(k, v.shape, v.dtype)

print("ground_truth_mask shape:", batch["ground_truth_mask"].shape)


## 7. Fine-tune the SAM mask decoder

The vision encoder and prompt encoder are frozen. Only the mask decoder is updated.  
This keeps training lighter and is usually enough when adapting SAM to a specific segmentation task.


In [ ]:
# Training loop.
# For this notebook the number of epochs is set low so the cell can run as a quick test.
# Increase num_epochs once the data pipeline is confirmed to work.

from transformers import SamModel
import torch
import monai
from torch.optim import Adam
from tqdm import tqdm
from statistics import mean
import torch.nn.functional as F
import os

model = SamModel.from_pretrained("facebook/sam-vit-base")

# Freeze the heavy parts of SAM. The mask decoder is the part we fine-tune.
for name, param in model.named_parameters():
    if name.startswith("vision_encoder") or name.startswith("prompt_encoder"):
        param.requires_grad_(False)

optimizer = Adam(model.mask_decoder.parameters(), lr=1e-5, weight_decay=0)
seg_loss = monai.losses.DiceCELoss(sigmoid=True, squared_pred=True, reduction="mean")

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.train()

num_epochs = 1

for epoch in range(num_epochs):
    epoch_losses = []

    for batch in tqdm(train_dataloader):
        pixel_values = batch["pixel_values"].to(device)
        input_boxes  = batch["input_boxes"].to(device).float()
        gt_masks     = batch["ground_truth_mask"].to(device).float()

        outputs = model(
            pixel_values=pixel_values,
            input_boxes=input_boxes,
            multimask_output=False
        )

        predicted_masks = outputs.pred_masks

        # Normalise output shape to (B, 1, H, W).
        # Hugging Face/SAM versions can differ by one singleton dimension.
        if predicted_masks.dim() == 3:
            predicted_masks = predicted_masks.unsqueeze(1)
        elif predicted_masks.dim() == 5 and predicted_masks.shape[2] == 1:
            predicted_masks = predicted_masks.squeeze(2)

        # Match the ground-truth mask size before computing the loss.
        if predicted_masks.shape[-2:] != gt_masks.shape[-2:]:
            predicted_masks = F.interpolate(
                predicted_masks,
                size=gt_masks.shape[-2:],
                mode="bilinear",
                align_corners=False
            )

        loss = seg_loss(predicted_masks, gt_masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_losses.append(loss.item())

    print(f"EPOCH: {epoch}")
    print(f"Mean loss: {mean(epoch_losses):.6f}")

# Save the fine-tuned weights to Google Drive.
save_dir = "/content/drive/MyDrive/Colab/models/SAM"
os.makedirs(save_dir, exist_ok=True)
ckpt_path = f"{save_dir}/blood_model_checkpoint.pth"
torch.save(model.state_dict(), ckpt_path)
print("Saved to:", ckpt_path)


## 8. Apply the trained model

The next cells load the saved checkpoint and test the model on either a random training patch or a larger external image.


In [ ]:
# Load the fine-tuned model checkpoint.
# Keep base_checkpoint the same as the SAM model used during training.

from transformers import SamModel, SamProcessor
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
base_checkpoint = "facebook/sam-vit-base"

processor = SamProcessor.from_pretrained(base_checkpoint)
model = SamModel.from_pretrained(base_checkpoint).to(device)

ckpt_path = "/content/drive/MyDrive/Colab/models/SAM/blood_model_checkpoint.pth"
state_dict = torch.load(ckpt_path, map_location=device)
model.load_state_dict(state_dict)

model.eval()
print("Loaded fine-tuned model from:", ckpt_path)


In [ ]:
# Test the model on one random patch from the dataset.
# This is a fast way to check whether the checkpoint gives reasonable masks.

import numpy as np
import random
import torch
import matplotlib.pyplot as plt

idx = random.randint(0, len(dataset) - 1)

test_image = dataset[idx]["image"]
ground_truth_mask = np.array(dataset[idx]["label"])
ground_truth_mask_bin = (ground_truth_mask > 0).astype(np.uint8)

prompt = get_bounding_box(ground_truth_mask_bin)
inputs = processor(test_image, input_boxes=[[prompt]], return_tensors="pt")
inputs = {k: v.to(device) for k, v in inputs.items()}
inputs["input_boxes"] = inputs["input_boxes"].float()

model.eval()
with torch.no_grad():
    outputs = model(**inputs, multimask_output=False)

pred_logits = outputs.pred_masks
if pred_logits.dim() == 5 and pred_logits.shape[2] == 1:
    pred_logits = pred_logits.squeeze(2)

pred_prob = torch.sigmoid(pred_logits).squeeze().detach().cpu().numpy()
pred_mask = (pred_prob > 0.5).astype(np.uint8)

fig, axes = plt.subplots(1, 4, figsize=(18, 5))

axes[0].imshow(np.array(test_image))
axes[0].set_title("Image")

axes[1].imshow(ground_truth_mask_bin, cmap="gray")
axes[1].set_title("Ground Truth Mask")

axes[2].imshow(pred_mask, cmap="gray")
axes[2].set_title("Predicted Mask")

axes[3].imshow(pred_prob)
axes[3].set_title("Probability Map")

for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])

plt.show()


## 9. Segment a full-size image

The model was trained on 256 × 256 patches, so a larger image is processed patch by patch.  
Overlapping patches are averaged to reduce hard borders between neighbouring predictions.


In [ ]:
# Optional reload cell.
# Use this if you restarted the runtime and only want to run the application part.

from transformers import SamModel, SamProcessor
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

base_checkpoint = "facebook/sam-vit-base"
processor = SamProcessor.from_pretrained(base_checkpoint)

model = SamModel.from_pretrained(base_checkpoint).to(device)
ckpt_path = "/content/drive/MyDrive/ColabNotebooks/models/SAM/blood_model_checkpoint.pth"
model.load_state_dict(torch.load(ckpt_path, map_location=device))
model.eval()

print("model loaded")


In [ ]:
# Load a full test image for segmentation.
# The image is converted to RGB because SAM expects three-channel input.

from PIL import Image
import numpy as np

img = Image.open("/content/drive/MyDrive/Colab/test_neu.png").convert("RGB")
new_img = np.array(img)

print(new_img.shape, new_img.dtype)


In [ ]:
# Store image dimensions and keep the NumPy array under a short variable name.

print("new_img shape:", new_img.shape)
H, W = new_img.shape[:2]
print("H, W:", H, W)

img = new_img


In [ ]:
# Predict a probability map for a full-size image using overlapping patches.
# Each patch is prompted with a full-image box because no ground-truth mask is available at inference time.

import numpy as np
from PIL import Image
import torch
import matplotlib.pyplot as plt

def pad_to_multiple(img, patch_size=256):
    """Pad an RGB image so its height and width are multiples of patch_size."""
    H, W = img.shape[:2]
    Hp = int(np.ceil(H / patch_size) * patch_size)
    Wp = int(np.ceil(W / patch_size) * patch_size)

    padded = np.pad(
        img,
        ((0, Hp - H), (0, Wp - W), (0, 0)),
        mode="constant",
        constant_values=0
    )
    return padded, (H, W)


def predict_patch_prob_full_box(patch_rgb_uint8, model, processor, device):
    """Run SAM on one RGB patch and return a foreground probability map."""
    pil_patch = Image.fromarray(patch_rgb_uint8).convert("RGB")

    # Full-patch prompt: [x_min, y_min, x_max, y_max].
    full_box = [[[0, 0, patch_rgb_uint8.shape[1] - 1, patch_rgb_uint8.shape[0] - 1]]]

    inputs = processor(pil_patch, input_boxes=full_box, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    inputs["input_boxes"] = inputs["input_boxes"].float()

    with torch.no_grad():
        outputs = model(**inputs, multimask_output=False)

    logits = outputs.pred_masks
    if logits.dim() == 5 and logits.shape[2] == 1:
        logits = logits.squeeze(2)

    prob = torch.sigmoid(logits).squeeze().detach().cpu().numpy().astype(np.float32)
    return prob


def predict_full_prob_overlap(img_rgb_uint8, model, processor, device,
                              patch_size=256, stride=128):
    """
    Segment a full image by sliding overlapping patches across it.

    Overlap is useful because patch edges are often less reliable. For pixels covered by
    multiple patches, the predicted probabilities are averaged.
    """
    padded, (H, W) = pad_to_multiple(img_rgb_uint8, patch_size)
    Hp, Wp = padded.shape[:2]

    prob_sum = np.zeros((Hp, Wp), dtype=np.float32)
    prob_cnt = np.zeros((Hp, Wp), dtype=np.float32)

    model.eval()
    for y0 in range(0, Hp - patch_size + 1, stride):
        for x0 in range(0, Wp - patch_size + 1, stride):
            patch = padded[y0:y0 + patch_size, x0:x0 + patch_size, :]
            prob = predict_patch_prob_full_box(patch, model, processor, device)

            prob_sum[y0:y0 + patch_size, x0:x0 + patch_size] += prob
            prob_cnt[y0:y0 + patch_size, x0:x0 + patch_size] += 1.0

    prob_avg = prob_sum / np.maximum(prob_cnt, 1e-6)
    return prob_avg[:H, :W]


# Run prediction on the full image.
full_prob = predict_full_prob_overlap(
    img,
    model,
    processor,
    device,
    patch_size=256,
    stride=128
)

# Convert probability map to a binary mask.
thresh = 0.6
full_mask = (full_prob > thresh).astype(np.uint8)

plt.figure(figsize=(18, 6))
plt.subplot(1, 3, 1)
plt.title("Image")
plt.imshow(img)
plt.axis("off")

plt.subplot(1, 3, 2)
plt.title("Probability map")
plt.imshow(full_prob)
plt.axis("off")

plt.subplot(1, 3, 3)
plt.title(f"Mask (threshold = {thresh})")
plt.imshow(full_mask, cmap="gray")
plt.axis("off")

plt.show()


In [ ]:
# Save the raw predicted mask.
# Values are converted from 0/1 to 0/255 so the file opens correctly as a black-white image.

from PIL import Image
import numpy as np

out_path = "/content/drive/MyDrive/Colab/test_image_mask_clean.png"
mask_to_save = (full_mask * 255).astype(np.uint8)

Image.fromarray(mask_to_save).save(out_path)
print("Saved mask to:", out_path)


## 10. Clean the predicted mask

The raw SAM mask can contain small islands, holes, or slightly rough borders.  
The cleanup below uses connected components and simple morphology to make the final mask easier to use for downstream analysis.


In [ ]:
# Post-process the predicted mask.
# Convention in this section: 0 = vessel/foreground, 255 = background.

import numpy as np
import cv2

def to_0_255(mask):
    """Convert a binary mask to uint8 values 0 and 255."""
    m = mask.astype(np.uint8)
    if m.max() == 1:
        m = m * 255
    return m


def remove_small_black_islands(mask_bw, max_black_area=200):
    """
    Remove small black foreground regions.

    This helps remove isolated segmentation noise while keeping larger vessel regions.
    """
    m = to_0_255(mask_bw)

    # Connected components expects foreground pixels, so black pixels are temporarily set to 255.
    fg = (m == 0).astype(np.uint8) * 255
    num, labels, stats, _ = cv2.connectedComponentsWithStats(fg, connectivity=8)

    cleaned_fg = np.zeros_like(fg, dtype=np.uint8)
    for lab in range(1, num):
        area = stats[lab, cv2.CC_STAT_AREA]
        if area >= max_black_area:
            cleaned_fg[labels == lab] = 255

    out = np.full_like(m, 255, dtype=np.uint8)
    out[cleaned_fg == 255] = 0
    return out


def fill_small_white_holes_in_black(mask_bw, max_hole_area=200):
    """
    Fill small white holes enclosed by black foreground.

    White regions touching the image border are kept, because those are background rather than holes.
    """
    m = to_0_255(mask_bw)
    holes = (m == 255).astype(np.uint8) * 255

    num, labels, stats, _ = cv2.connectedComponentsWithStats(holes, connectivity=8)
    H, W = m.shape
    out = m.copy()

    for lab in range(1, num):
        area = stats[lab, cv2.CC_STAT_AREA]
        if area > max_hole_area:
            continue

        x = stats[lab, cv2.CC_STAT_LEFT]
        y = stats[lab, cv2.CC_STAT_TOP]
        w = stats[lab, cv2.CC_STAT_WIDTH]
        h = stats[lab, cv2.CC_STAT_HEIGHT]

        touches_border = (x == 0) or (y == 0) or (x + w == W) or (y + h == H)

        if not touches_border:
            out[labels == lab] = 0

    return out


def morph_smooth(mask_bw, k_close=5, k_open=0):
    """
    Smooth the binary mask using morphological operations.

    Closing fills small gaps. Opening removes small specks. Kernel sizes should be tuned
    depending on the image resolution and the typical vessel thickness.
    """
    m = to_0_255(mask_bw)
    fg = (m == 0).astype(np.uint8) * 255

    if k_open and k_open > 0:
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k_open, k_open))
        fg = cv2.morphologyEx(fg, cv2.MORPH_OPEN, kernel)

    if k_close and k_close > 0:
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k_close, k_close))
        fg = cv2.morphologyEx(fg, cv2.MORPH_CLOSE, kernel)

    out = np.full_like(m, 255, dtype=np.uint8)
    out[fg == 255] = 0
    return out


# Cleanup parameters. These values are image-dependent and should be tuned visually.
mask_bw = to_0_255(full_mask)
mask_bw = remove_small_black_islands(mask_bw, max_black_area=1500)
mask_bw = fill_small_white_holes_in_black(mask_bw, max_hole_area=300)
mask_bw = morph_smooth(mask_bw, k_close=5, k_open=0)

mask_clean_bw = mask_bw


In [ ]:
# Compare the raw mask with the cleaned result.

import matplotlib.pyplot as plt

plt.figure(figsize=(16, 6))
plt.subplot(1, 2, 1)
plt.title("Raw mask")
plt.imshow(to_0_255(full_mask), cmap="gray")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.title("Cleaned mask")
plt.imshow(mask_clean_bw, cmap="gray")
plt.axis("off")

plt.show()


In [ ]:
# Save the final cleaned mask.

from PIL import Image

out_path = "/content/drive/MyDrive/Colab/test_image_mask_clean_final.png"
Image.fromarray(mask_clean_bw).save(out_path)

print("Saved:", out_path)
